# 예측 구간 실습

**Prediction Interval · 신뢰 구간**

새 관측값이 들어갈 것으로 기대되는 범위. 평균의 신뢰 구간보다 넓다.

소재 분야에서 이해하기: 예측 강도에 90% 구간을 함께 보고한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [scikit-learn 가우시안 프로세스 문서](https://scikit-learn.org/stable/modules/gaussian_process.html)

## 1. 평균의 구간과 관측의 구간은 다릅니다

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

x = rng.uniform(0, 10, 200)
y = 3 + 1.5 * x + rng.normal(0, 2.0, 200)
plt.scatter(x, y, s=12); plt.xlabel('x'); plt.ylabel('y'); plt.show()

In [ ]:
import numpy.linalg as la

design = np.column_stack([np.ones_like(x), x])
coefficients, *_ = la.lstsq(design, y, rcond=None)
residual = y - design @ coefficients
sigma2 = residual @ residual / (len(x) - 2)
covariance = sigma2 * la.inv(design.T @ design)

grid = np.linspace(0, 10, 200)
grid_design = np.column_stack([np.ones_like(grid), grid])
mean = grid_design @ coefficients
mean_var = np.einsum('ij,jk,ik->i', grid_design, covariance, grid_design)
confidence = 1.96 * np.sqrt(mean_var)
prediction = 1.96 * np.sqrt(mean_var + sigma2)

plt.scatter(x, y, s=10, c='lightgray')
plt.plot(grid, mean, 'b-')
plt.fill_between(grid, mean - confidence, mean + confidence, alpha=0.4, label='95% CI of the mean')
plt.fill_between(grid, mean - prediction, mean + prediction, alpha=0.2, label='95% prediction interval')
plt.legend(); plt.xlabel('x'); plt.ylabel('y'); plt.show()

## 2. 구간이 실제로 맞는지 검증

In [ ]:
x_new = rng.uniform(0, 10, 5000)
y_new = 3 + 1.5 * x_new + rng.normal(0, 2.0, 5000)
new_design = np.column_stack([np.ones_like(x_new), x_new])
new_mean = new_design @ coefficients
new_var = np.einsum('ij,jk,ik->i', new_design, covariance, new_design)
inside_ci = np.mean(np.abs(y_new - new_mean) < 1.96 * np.sqrt(new_var))
inside_pi = np.mean(np.abs(y_new - new_mean) < 1.96 * np.sqrt(new_var + sigma2))
print('새 관측이 평균의 신뢰구간에 들어간 비율 %.3f (목표가 아님)' % inside_ci)
print('새 관측이 예측구간에 들어간 비율        %.3f (0.95 에 가까워야 정상)' % inside_pi)

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#prediction-interval)을 여세요.